In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt

In [2]:
class GridWorld:
    def __init__(self):
        self.size = 5
        self.start = (0, 0)
        self.goal = (4, 4)
        self.obstacles = {(1, 1), (2, 2), (3, 1), (3, 3), (1, 3)}

        # Actions: UP, DOWN, LEFT, RIGHT
        self.actions = {
            0: (-1, 0),  # UP
            1: (1, 0),   # DOWN
            2: (0, -1),  # LEFT
            3: (0, 1)    # RIGHT
        }

        self.reset()

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        r, c = self.state
        dr, dc = self.actions[action]
        nr, nc = r + dr, c + dc

        # Check boundary condition
        if nr < 0 or nr >= self.size or nc < 0 or nc >= self.size:
            return self.state, -1, False

        # Check obstacle
        if (nr, nc) in self.obstacles:
            return self.state, -10, False

        # Move to new state
        new_state = (nr, nc)

        # Goal reached
        if new_state == self.goal:
            return new_state, 100, True

        # Normal step
        return new_state, -1, False

In [3]:
env = GridWorld()

In [4]:
alpha = 0.1
gamma = 0.99
epsilon = 0.1
episodes = 100

In [5]:
Q = np.zeros((5, 5, 4))

In [6]:
episode_rewards = []

In [7]:
def choose_action(state):
    if random.uniform(0, 1) < epsilon:
        return random.randint(0, 3)
    else:
        r, c = state
        return np.argmax(Q[r, c])

In [ ]:
for ep in range(episodes):
    state = env.reset()
    total_reward = 0

    while True:
        r, c = state
        action = choose_action(state)

        next_state, reward, done = env.step(action)
        nr, nc = next_state

        # Q-learning update
        Q[r, c, action] = Q[r, c, action] + alpha * (
            reward + gamma * np.max(Q[nr, nc]) - Q[r, c, action]
        )

        state = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)

In [ ]:
policy_grid = [["" for _ in range(5)] for _ in range(5)]

In [ ]:
arrows = {0: "↑", 1: "↓", 2: "←", 3: "→"}

In [ ]:
for r in range(5):
    for c in range(5):
        if (r, c) in env.obstacles:
            policy_grid[r][c] = "X"
        elif (r, c) == env.goal:
            policy_grid[r][c] = "G"
        elif (r, c) == env.start:
            policy_grid[r][c] = "S"
        else:
            best_action = np.argmax(Q[r, c])
            policy_grid[r][c] = arrows[best_action]

In [ ]:
print("\nOptimal Policy:")
for row in policy_grid:
    print(row)

In [ ]:
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Episode Rewards Over Time")
plt.grid()
plt.show()